# M305 - APIs y Automatización

> Módulo III - Python para la inteligencia artificial
>
> Tema 5: Integración de APIs y flujos de datos automatizados

Contenido:

1. Contrucción del API
2. Proveedor de datos
3. Proveedor de gráficas
4. Proveedor de interfaces web

Objetivo: Construir un API para brindar datos e inferencias

In [3]:
# ! python -m pip install flask

## 1. Contrucción del API

Un **API** (*Application Program Interface*) es una interfaz que provee recursos de forma estándar a tráves de los protocolos de red, principalmente el **HTTP** (*Hyper-Text Transfer Protocol*).

Un *hípertexto* se puede entender como una codificación de un recurso en texto extendido, por ejemplo, un texto plano (`text/plain`), un texto con formato HTML/XML/CSV (`text/html`, `text/xml`, `text/csv`), una imagen (`image/png`, `image/jpeg`, `image/gif`, `image/svg`) o datos binarios sin una codificación específica (`octect-stream`, `binary-stream`). La codificación se conoce como el **Mime-Type** (el tipo asociado al hípertexto), y este permite una representación específica en el cliente/navegador (**JSON** -> `application/json`), esto permite que el navegador o el cliente interprete adecuadamente los datos y los represente de la mejor forma, por ejemplo, el navegador toma los datos HTML y construye una interfaz, el navegador toma los datos PDF y muestra un visualizador con herramientas sobre el documento o toma imagen y la visualiza directamente.

El hípertexto viaja como texto a través de protocolo HTTP bajo un esquema de una URL, que se divide en una petición (*Request*) y una respuesta (*Response*). Este mecanismo permite establecer un programa que atiende de forma paralela solicitudes/peticiones de recursos, donde cada recurso es accedido mediante una *URL* y determina cabeceras que indican el estatus de entrada y salida, por ejemplo, el tipo de formato de entrada de datos, el método solicitado (*GET*, *POST*, *PUT*, *PATCH*, *DELETE*, etc.), y finalmente devuelve cabeceras de salida con un estatus (HTTP: `200 Ok`, `400 Bad request`, `404 Not found`, `500 Internal server error`, `401 Access denied`, `403 Forbidden`, etc.). 

Con esto el API estructura un mecanismo petición-respuesta a recursos codificados mediante una URL.

Este programa que atiende las solicitudes se conoce como el servidor (de recursos) y se establece o ejecuta en un PID (ID del programa en el sistema operativo) y reservan un puerto específico (Puerto de Red) para escuchar en ese puerto todas las solicitudes que generalmente son transformadas de los protocolos físicos TCP/UDP a HTTP. Por ejemplo, en el puerto 80, generalmente conocido internamente como el puerto principal o en el puerto 443 conocido como el puerto certificado, se monta el servidor principal, pero en APIs general se usan otros puertos como el `3000`, `5000`, `8000`, etc.

Algunos puertos famosos no pueden ser utilizados porque los podría ocupar otro programa como una base de datos (*MySQL* -> `3306` o *MongoDB* -> `27017`), o programas de utilería (*SSH* -> `22`, `FTP` -> `21/23`, etc.).

Un script de Python puede montar a tráves de *Flask* un servidor y comenzar a escuchar peticiones desde el navegador o un cliente (*CURL*, *Postman*, o alguna aplicación en general), y que tenga acceso directo a la IP y el PUERTO o el dominio enlazado (*Ngnix* o *Apache*).

La idea principal es que mientras el programa de python esté en ejecución, los recursos servidos en cada ruta (URL) escuchada serán accesibles en la red, dentro de la misma computadora generalmente usaremos `http://localhost:<puerto>` o `http://127.0.0.1:<puerto>`, y al menos que usemos el puerto 80 y nadie más lo use (otro programa) podremos simplemente acceder a `http://localhost` o `http://127.0.0.1`. Fuera de la computadora, se puede acceder con `http://<ip>:<puerto>` o `http(s)://<dominio>`.

Generalmente un servidor opera las 24 horas (una computadora encendida todo el tiempo con una red estable, por ejemplo, una IP Pública o un Dominio Público).

Mientras el programa esté ejecutándose podremos acceder a los recursos escuchados y atendidos desde un navegador o el comando `curl`.

In [7]:
# ! ifconfig | grep 192.168

# 	inet 192.168.1.83 netmask 0xffffff00 broadcast 192.168.1.255

In [1]:
from flask import Flask

app = Flask(__name__)

# Cuando se ejecuta se mantiene en ejecución
# y podemos consumir sus recursos en http://localhost:5000 por defecto
# !!! Dentro de una libreta debemos detener manualmente al servidor si hacemos cambios
app.run(host="0.0.0.0", port=3000) 
# host: Define el acceso (máscara de red -> 0.0.0.0 / Todas las IPs externas)
#       Para acceso solo en su computadora (por defecto) host="localhost" o host="127.0.0.1"
# Cambiamos el puerto

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:3000
 * Running on http://172.19.0.2:3000
Press CTRL+C to quit
127.0.0.1 - - [19/Nov/2026 18:11:31] "GET / HTTP/1.1" 404 -


In [3]:
from flask import Flask

app = Flask(__name__)

@app.get("/") # Ruta del recurso http://localhost:3000/
def main():
    return "Hola mundo desde Flask 🐳" # Mime-Type: text/plain

app.run(port=3000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:3000
Press CTRL+C to quit
127.0.0.1 - - [19/Nov/2026 18:21:16] "GET / HTTP/1.1" 200 -


## 2. Proveedor de datos

In [5]:
from flask import Flask

app = Flask(__name__)

@app.get("/") # text/plain
def main():
    return "v1.0.0"

@app.get("/datos1") # application/json
def datos1():
    return {
        "nombre": "Ana",
        "edad": 34,
        "sexo": "mujer",
        "salario": 15800,
        "casado": True
    }

app.run(port=3000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:3000
Press CTRL+C to quit
127.0.0.1 - - [19/Nov/2026 18:24:15] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 18:24:28] "GET /datos1 HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 18:25:04] "GET /datos1 HTTP/1.1" 200 -


In [7]:
from flask import Flask

import pandas

# --- DATOS ---

titanic = pandas.read_csv("./conjuntos/titanic.csv")
iris = pandas.read_csv("./conjuntos/iris.csv")

# --- SERVIDOR ---

app = Flask(__name__)

@app.get("/") # text/plain
def main():
    return "v1.0.1"

@app.get("/datos1") # application/json
def datos1():
    return {
        "nombre": "Ana",
        "edad": 34,
        "sexo": "mujer",
        "salario": 15800,
        "casado": True
    }

@app.get("/datos/titanic") # application/json
def datos_titanic():
    return titanic.to_json()

@app.get("/datos/iris") # application/json
def datos_iris():
    return iris.to_json()

app.run(port=3000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:3000
Press CTRL+C to quit
127.0.0.1 - - [19/Nov/2026 18:30:59] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 18:31:35] "GET /datos/titanic HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 18:31:55] "GET /datos/iris HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 18:34:45] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 18:35:34] "GET /datos/titanic HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 18:35:55] "GET /datos/titanic HTTP/1.1" 200 -


In [8]:
from flask import Flask
from flask import request

import pandas

# --- DATOS ---

titanic = pandas.read_csv("./conjuntos/titanic.csv")

# --- SERVIDOR ---

app = Flask(__name__)

@app.get("/") # text/plain
def main():
    return "v1.0.2"

@app.get("/datos/titanic") # application/json
def datos_titanic():
    survived = request.args.get("survived", default=None, type=int)

    if not survived is None:
        return titanic[titanic["Survived"] == survived].to_json()
    
    return titanic.to_json()

app.run(port=3000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:3000
Press CTRL+C to quit
127.0.0.1 - - [19/Nov/2026 18:39:03] "GET /datos/titanic?survived=1 HTTP/1.1" 200 -


In [16]:
from flask import Flask
from flask import request

import pandas

# --- DATOS ---

titanic = pandas.read_csv("./conjuntos/titanic.csv")

# --- SERVIDOR ---

app = Flask(__name__)

@app.get("/") # text/plain
def main():
    return "v1.0.3"

@app.get("/datos/titanic") # application/json
def datos_titanic():
    columns = request.args.get("columns", default=None, type=str)
    survived = request.args.get("survived", default=None, type=int)
    age_min = request.args.get("age_min", default=None, type=int)
    age_max = request.args.get("age_max", default=None, type=int)
    pclass = request.args.get("pclass", default=None, type=int)
    fare_min = request.args.get("fare_min", default=None, type=float)
    fare_max = request.args.get("fare_max", default=None, type=float)
    cabin = request.args.get("cabin", default=None, type=bool)
    embarked = request.args.get("embarked", default=None, type=str)
    sibsp_min = request.args.get("sibsp_min", default=None, type=int)
    sibsp_max = request.args.get("sibsp_max", default=None, type=int)
    parch_min = request.args.get("parch_min", default=None, type=int)
    parch_max = request.args.get("parch_max", default=None, type=int)

    tabla = titanic.copy()

    if not survived is None:
        tabla = tabla[tabla["Survived"] == survived]
    
    if not age_min is None:
        tabla = tabla[tabla["Age"] >= age_min]
    if not age_max is None:
        tabla = tabla[tabla["Age"] <= age_max]
    
    if not pclass is None:
        tabla = tabla[tabla["Pclass"] == pclass]
    
    if not fare_min is None:
        tabla = tabla[tabla["Fare"] >= fare_min]
    if not fare_max is None:
        tabla = tabla[tabla["Fare"] <= fare_max]
    
    if not cabin is None:
        tabla = tabla[(~(tabla["Cabin"].isna())) == cabin]
    
    if not embarked is None:
        tabla = tabla[tabla["Embarked"] == embarked]

    if not sibsp_min is None:
        tabla = tabla[tabla["SibSp"] >= sibsp_min]
    if not sibsp_max is None:
        tabla = tabla[tabla["SibSp"] <= sibsp_max]
    if not parch_min is None:
        tabla = tabla[tabla["Parch"] >= parch_min]
    if not parch_max is None:
        tabla = tabla[tabla["Parch"] <= parch_max]

    if not columns is None:
        columns = columns.split(",")
        tabla = tabla[columns]
        
    return tabla.to_json()

app.run(port=3000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:3000
Press CTRL+C to quit
127.0.0.1 - - [19/Nov/2026 18:56:35] "GET /datos/titanic?columns=Age,Survived&age_max=15&embarked=S HTTP/1.1" 200 -


In [17]:
import pandas
import json

# curl 'http://localhost:3000/datos/titanic?columns=Age,Survived&age_max=15&embarked=S'

frames = json.loads('{"Age":{"7":2.0,"10":4.0,"14":14.0,"24":8.0,"50":7.0,"58":5.0,"59":11.0,"63":4.0,"78":0.83,"119":2.0,"147":9.0,"164":1.0,"165":9.0,"172":1.0,"182":9.0,"183":1.0,"184":4.0,"193":3.0,"205":2.0,"233":5.0,"237":8.0,"261":3.0,"297":2.0,"305":0.92,"340":2.0,"348":3.0,"374":3.0,"386":1.0,"407":3.0,"419":10.0,"435":14.0,"445":4.0,"446":13.0,"479":2.0,"480":9.0,"489":9.0,"530":2.0,"535":7.0,"541":9.0,"542":11.0,"549":8.0,"618":4.0,"634":9.0,"642":2.0,"683":14.0,"686":14.0,"689":15.0,"720":6.0,"750":4.0,"751":6.0,"755":0.67,"777":5.0,"788":1.0,"802":11.0,"813":6.0,"819":10.0,"824":2.0,"831":0.83,"850":4.0,"869":4.0},"Survived":{"7":0,"10":1,"14":0,"24":0,"50":0,"58":1,"59":0,"63":0,"78":1,"119":0,"147":0,"164":0,"165":1,"172":1,"182":0,"183":1,"184":1,"193":1,"205":0,"233":1,"237":1,"261":1,"297":0,"305":1,"340":1,"348":1,"374":0,"386":0,"407":1,"419":0,"435":1,"445":1,"446":1,"479":1,"480":0,"489":1,"530":1,"535":1,"541":0,"542":0,"549":1,"618":1,"634":0,"642":0,"683":0,"686":0,"689":1,"720":1,"750":1,"751":1,"755":1,"777":1,"788":1,"802":1,"813":0,"819":0,"824":0,"831":1,"850":0,"869":1}}')

pandas.DataFrame(frames)

,Age,Survived
7,2.00,0
10,4.00,1
14,14.00,0
24,8.00,0
50,7.00,0
58,5.00,1
59,11.00,0
63,4.00,0
78,0.83,1
119,2.00,0


## 3. Proveedor de gráficas

In [ ]:
from flask import Flask
from flask import request
from flask import send_file

import pandas

from matplotlib import pyplot
import seaborn

import io

# --- DATOS ---

titanic = pandas.read_csv("./conjuntos/titanic.csv")

# --- SERVIDOR ---

app = Flask(__name__)

@app.get("/") # text/plain
def main():
    return "v1.0.3"

@app.get("/graficas/titanic/sexo_vs_supervivencia") # application/json
def graficas_titanic_sexo_vs_supervivencia():
    M = titanic[["Sex", "Survived"]].groupby(["Sex", "Survived"]).size().unstack()
    R1 = M.divide(M.sum(axis=1), axis=0)
    R2 = M.divide(M.sum(axis=0), axis=1)

    figure, axis = pyplot.subplots(1, 3, figsize=(40, 10))

    seaborn.heatmap(M, cmap="Greens", annot=True, fmt=".0f", ax=axis[0])
    seaborn.heatmap(R1, cmap="Blues", annot=True, fmt=".1f", ax=axis[1])
    seaborn.heatmap(R2, cmap="Reds", annot=True, fmt=".1f", ax=axis[2])

    # 1. Guardar la gráfica en un buffer de memoria
    buffer = io.BytesIO()
    figure.savefig(buffer, format="png", bbox_inches="tight")
    
    # 2. Liberar la memoria de la figura de Matplotlib
    pyplot.close(figure)
    
    # 3. Regresar el puntero al inicio del archivo en memoria
    buffer.seek(0)

    # 4. Devolver la imagen con el mimetype adecuado
    return send_file(buffer, mimetype="image/png")

# curl --output sexo_vs_supervivencia.png localhost:3000/graficas/titanic/sexo_vs_supervivencia
app.run(port=3000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:3000
Press CTRL+C to quit
127.0.0.1 - - [19/Nov/2026 19:03:20] "GET /graficas/titanic/sexo_vs_supervivencia HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 19:04:01] "GET /graficas/titanic/sexo_vs_supervivencia HTTP/1.1" 200 -


## 4. Proveedor de interfaces web